In [ ]:
import sys
from pathlib import Path

NOTEBOOK_PATH_CANDIDATES = [Path.cwd(), Path.cwd() / "AgentWorkshop" / "Notebook"]
for candidate in NOTEBOOK_PATH_CANDIDATES:
    if (candidate / "workshop_bootstrap.py").exists():
        resolved_candidate = str(candidate.resolve())
        if resolved_candidate not in sys.path:
            sys.path.insert(0, resolved_candidate)

from workshop_bootstrap import build_workshop_config

CONFIG_OVERRIDES = {
    "resource_group_name": "",
    "location": "",
    "subscription_id": "",
    "foundry_account_name": "",
    "foundry_project_name": "",
    "foundry_project_endpoint": "",
    "search_service_name": "",
    "storage_account_name": "",
    "application_insights_name": "",
    "model_zone": "",
}

config = build_workshop_config({k: v for k, v in CONFIG_OVERRIDES.items() if v})
config.show()

# Workshop 3: Agents

This notebook mirrors docs/agents.md and creates the main workshop agents using code.

Agents created in this notebook:
- stats-for-coffee (Code Interpreter with CSV files)
- coffee-research-agent (retrieval-focused instruction set)

In [ ]:
# Uncomment this cell in a clean kernel.
# %pip install --quiet "azure-ai-projects>=2.0.0" azure-identity

In [ ]:
import os
from pathlib import Path

from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import AutoCodeInterpreterToolParam, CodeInterpreterTool, PromptAgentDefinition

if not config.foundry_project_endpoint:
    raise ValueError("Set AZURE_AI_PROJECT_ENDPOINT or provide foundry_project_endpoint in CONFIG_OVERRIDES.")

CHAT_DEPLOYMENT_NAME = os.getenv("CHAT_DEPLOYMENT_NAME", "").strip()
if not CHAT_DEPLOYMENT_NAME:
    raise ValueError("Set CHAT_DEPLOYMENT_NAME in your environment.")

DATASET_GENERAL = Path("../../data/Coffee/CoffeeCSV/GeneralHealth/synthetic_mental_health_dataset.csv").resolve()
DATASET_LARGE = Path("../../data/Coffee/CoffeeCSV/mentalHealth/synthetic_coffee_health_10000.csv").resolve()

if not DATASET_GENERAL.exists() or not DATASET_LARGE.exists():
    raise FileNotFoundError("Required CSV files were not found in data/Coffee/CoffeeCSV.")

project = AIProjectClient(endpoint=config.foundry_project_endpoint, credential=DefaultAzureCredential())
openai = project.get_openai_client()

In [ ]:
with DATASET_GENERAL.open("rb") as file_handle:
    general_file = openai.files.create(purpose="assistants", file=file_handle)

with DATASET_LARGE.open("rb") as file_handle:
    large_file = openai.files.create(purpose="assistants", file=file_handle)

print("Uploaded files:")
print("- general:", general_file.id)
print("- large:  ", large_file.id)

In [ ]:
STATS_AGENT_NAME = "stats-for-coffee"
RESEARCH_AGENT_NAME = "coffee-research-agent"

STATS_AGENT_INSTRUCTIONS = """
You are a rigorous data-analysis agent for a scientific workshop in Microsoft Foundry.

Primary mission:
- Analyze two workshop CSV files using Code Interpreter.
- Explain findings in plain English and show the exact calculation logic.
- Generate charts when they help understanding.
- Be explicit about uncertainty, synthetic-data limitations, and non-causal interpretations.

Behavioral rules:
- Always inspect schema first before calculating.
- Use Python for numeric, statistical, plotting, grouping, filtering, and transformation tasks.
- Do not claim causation from correlation.
- If asked to compare both datasets, do thematic comparison unless a shared key is provided.

Preferred output format:
1. Question
2. Datasets and columns
3. Method
4. Results
5. Interpretation
6. Caveats
"""

RESEARCH_AGENT_INSTRUCTIONS = """
You are Coffee Research Agent, an academic research assistant.

Mission:
Provide evidence-based, citation-rich answers about coffee and health with clear uncertainty reporting.

Grounding rules:
- Prioritize scientific health evidence.
- Treat CSV-based observations as exploratory and non-clinical.
- Do not invent facts, statistics, or citations.
- Do not infer causation from correlation without explicit evidence.
"""

stats_agent = project.agents.create_version(
    agent_name=STATS_AGENT_NAME,
    definition=PromptAgentDefinition(
        model=CHAT_DEPLOYMENT_NAME,
        instructions=STATS_AGENT_INSTRUCTIONS,
        tools=[
            CodeInterpreterTool(
                container=AutoCodeInterpreterToolParam(
                    file_ids=[general_file.id, large_file.id]
                )
            )
        ],
    ),
    description="Workshop statistical analysis agent.",
)

research_agent = project.agents.create_version(
    agent_name=RESEARCH_AGENT_NAME,
    definition=PromptAgentDefinition(
        model=CHAT_DEPLOYMENT_NAME,
        instructions=RESEARCH_AGENT_INSTRUCTIONS,
        tools=[],
    ),
    description="Workshop evidence synthesis agent.",
)

print("Created agent versions:")
print("- stats agent:   ", stats_agent.name, stats_agent.version)
print("- research agent:", research_agent.name, research_agent.version)

In [ ]:
conversation = openai.conversations.create()

def ask_agent(agent_name: str, prompt: str):
    return openai.responses.create(
        conversation=conversation.id,
        input=prompt,
        extra_body={
            "agent_reference": {
                "name": agent_name,
                "type": "agent_reference",
            }
        },
    )

prompt_text = "Start by inspecting both datasets and provide schema, missingness, and quality issues."
stats_response = ask_agent(STATS_AGENT_NAME, prompt_text)
stats_response

## Monitoring

Use Application Insights from baseline deployment for agent monitoring in Foundry portal.

## Next
Continue with 05_multi_agent_workshop.ipynb for sequential multi-agent orchestration.